# SmartRide: from Statistics to Machine Learning and Optimization

This notebook builds a small ride-hailing matching simulator. All records are synthetic. We will separate two ideas: **prediction** (machine learning estimates pickup ETA) and **decision-making** (optimization assigns drivers to riders).

Learning goals: generate data, calculate Haversine distance, train/evaluate a regression model, create an ETA matrix, and solve a batch assignment problem.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))
# If running from the repository root instead, uncomment the next line:
# sys.path.insert(0, str(Path.cwd() / 'src'))

import numpy as np
import pandas as pd
from smartride.data import generate_marketplace
from smartride.features import haversine_km, pairwise_distance_matrix
from smartride.model import train_eta_model
from smartride.matching import build_eta_matrix, optimize_assignments, greedy_nearest_assignments

## 1. Generate a synthetic marketplace

We create rider and driver coordinates around a generic Nairobi-area centre, plus historical trips. These are randomly generated examples, not real locations. The historical pickup time is generated from distance, traffic, weather, rush-hour effect and random noise.

In [ ]:
riders, drivers, trips = generate_marketplace(n_riders=12, n_drivers=15, seed=42)
display(riders.head())
display(drivers.head())
display(trips.head())
print('Historical rows:', len(trips))

## 2. Geographic distance: Haversine formula

Latitude and longitude are angles on a sphere. Haversine calculates great-circle distance, which is useful as a first geographic feature. It is **not** the same as driving distance because roads do not follow straight lines.

In [ ]:
# Same point should have zero distance
haversine_km(-1.2864, 36.8172, -1.2864, 36.8172)

distance_matrix = pairwise_distance_matrix(riders, drivers)
print('Matrix shape (riders x drivers):', distance_matrix.shape)
pd.DataFrame(distance_matrix, index=riders.rider_id, columns=drivers.driver_id).round(2).head()

## 3. Train a pickup ETA regression model

Our target is `pickup_eta_min`, a continuous numerical variable, so this is a regression task. A Random Forest learns non-linear relationships from the synthetic historical trips. We reserve a holdout test set so evaluation is on examples not used for fitting.

In [ ]:
model, metrics = train_eta_model(trips, random_state=42)
print(f"Holdout MAE:  {metrics['mae_min']:.2f} minutes")
print(f"Holdout RMSE: {metrics['rmse_min']:.2f} minutes")

# MAE is the average absolute prediction error. RMSE penalizes large errors more strongly.

## 4. Predict every possible rider-driver pickup ETA

For each possible pair, we combine its geographic distance with the scenario's hour, traffic and weather indices. The model predicts a cost (pickup minutes). This produces a matrix where rows are riders and columns are drivers.

In [ ]:
eta_matrix = build_eta_matrix(riders, drivers, model, hour=17, traffic_index=0.6, weather_index=0.2)
eta_table = pd.DataFrame(eta_matrix, index=riders.rider_id, columns=drivers.driver_id)
eta_table.round(1).head()

## 5. Optimize the batch assignment

The Hungarian algorithm chooses one-to-one rider-driver pairs that minimize the total predicted pickup time. This is a combinatorial optimization algorithm, not machine learning. When there are more drivers than riders, some drivers remain unassigned.

In [ ]:
optimal = optimize_assignments(eta_matrix, riders, drivers)
greedy = greedy_nearest_assignments(eta_matrix, riders, drivers)
display(optimal.round(2))
print('Optimized total ETA:', round(optimal.predicted_pickup_min.sum(), 2), 'minutes')
print('Greedy total ETA:   ', round(greedy.predicted_pickup_min.sum(), 2), 'minutes')

## 6. Interpret the result carefully

The global assignment objective can outperform a sequential greedy baseline, but it is not guaranteed to do so for every rider ordering or cost matrix. The objective here only minimizes the sum of predicted pickup ETAs. A real dispatch system has additional constraints and goals, including road-network travel time, driver acceptance, cancellation risk, service classes, fairness, earnings and system latency.

### Check your understanding
1. Why is pickup ETA a regression target rather than a classification label?
2. What does one row and one column of the ETA matrix represent?
3. Why is the Hungarian algorithm optimization rather than machine learning?
4. What important real-world information is missing from our synthetic dataset?

**Next improvement:** use a public, documented trip dataset and compare a linear regression baseline with Random Forest using cross-validation.